In [ ]:
# 1) Install dependencies (Internet must be ON)
import sys, subprocess
def pipi(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipi("flaml", "imbalanced-learn")  # lightgbm + xgboost (GPU-ready) ship with Kaggle
print("dependencies installed")

In [ ]:
# 2) Imports + configuration
import os, time, warnings
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score, recall_score,
    precision_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
    precision_recall_curve, roc_curve, auc)
from scipy.stats import friedmanchisquare
warnings.filterwarnings("ignore")

OUT = "/kaggle/working/results"
FIG = os.path.join(OUT, "figures"); SCORES = os.path.join(OUT, "scores")
for d in (OUT, FIG, SCORES): os.makedirs(d, exist_ok=True)
RESULTS_CSV = os.path.join(OUT, "results.csv")

COST_FN, COST_FP = 5.0, 1.0   # missing a positive (fraud/default) is 5x costlier than a false alarm
SK_N_ITER = 25
ESTIMATORS = ["lgbm", "rf", "extra_tree"]
STRATEGIES = ["S0_default", "S1_smote", "S2_objective", "S3_classweight",
              "S4_threshold", "S5_combined", "S6_full"]
# Save EVERY figure as BOTH .png (raster preview) and .pdf (vector, for the thesis)
_orig_fig_savefig = plt.Figure.savefig
def _savefig_both(self, fname, *a, **k):
    _orig_fig_savefig(self, fname, *a, **k)
    if isinstance(fname, str) and fname.lower().endswith(".png"):
        kk = {n: v for n, v in k.items() if n != "dpi"}
        _orig_fig_savefig(self, fname[:-4] + ".pdf", *a, **kk)
plt.Figure.savefig = _savefig_both
print("config ready ->", OUT, "| figures will be saved as .png AND .pdf")

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu if gpu else "none detected; use CPU engines flaml or sklearn")
print("Kaggle: Settings > Accelerator > GPU T4 x2 to enable the GPU engine.")


In [ ]:
# 3) Datasets — load ALL from /kaggle/input if present, else OpenML
import os
DATASETS = {
    "credit_card_fraud": 1597,   # ~0.17% minority
    "bank_marketing":    1461,   # ~11.7%
    "churn":             40701,  # ~14.1%
    "default_taiwan":    42477,  # ~22.1%
    "adult_income":      1590,   # ~23.9%
    "credit_g":          31,     # ~30.0%
    "australian_credit": 40981,  # ~44.5%
    "credit_approval":   29,     # ~44.5%
}

def _kaggle_index():
    idx = {}
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                idx.setdefault(f.lower(), os.path.join(root, f))
    return idx
KIDX = _kaggle_index()
print("files found in /kaggle/input:", len(KIDX))

def _find_local(name):
    cands = [name + ".parquet", name + ".csv"]
    if name == "credit_card_fraud":
        cands += ["creditcard.csv"]
    for c in cands:
        if c.lower() in KIDX:
            return KIDX[c.lower()]
    return None

def _to_binary_min_pos(y):
    y = y.astype(str); lab = y.value_counts().idxmin()
    return (y == lab).astype(int).values, str(lab)

def _read_local(path):
    df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
    if "target" in df.columns:
        y = df["target"].astype(int).values; X = df.drop(columns=["target"])
    elif "Class" in df.columns:
        y = df["Class"].astype(int).values; X = df.drop(columns=["Class"])
    else:                                  # last column is the label
        col = df.columns[-1]; y, _ = _to_binary_min_pos(df[col]); X = df.drop(columns=[col])
    return X, y

def load_dataset(name, max_rows=120000, random_state=0):
    path = _find_local(name)
    if path is not None:
        X, y = _read_local(path); src = "input:" + os.path.basename(path)
    else:
        b = fetch_openml(data_id=DATASETS[name], as_frame=True, parser="pandas")
        X = b.data.copy(); y, _ = _to_binary_min_pos(b.target); src = "openml"
    X = X.dropna(axis=1, how="all")
    if len(y) > max_rows:
        X, _, y, _ = train_test_split(X, y, train_size=max_rows, stratify=y, random_state=random_state)
        X = X.reset_index(drop=True)
    info = {"name": name, "source": src, "n_rows": int(len(y)), "n_features": int(X.shape[1]),
            "minority_rate": float(np.mean(y)), "n_minority": int(np.sum(y == 1))}
    return X, y, info

# quick check
_X, _y, _i = load_dataset("credit_g"); print(_i)

In [ ]:
# 4) Metrics + asymmetric cost
def g_mean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp/(tp+fn) if (tp+fn) else 0.0; tnr = tn/(tn+fp) if (tn+fp) else 0.0
    return float(np.sqrt(tpr*tnr))

def total_cost(y_true, y_pred, cost_fn=COST_FN, cost_fp=COST_FP):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return float(fn*cost_fn + fp*cost_fp)

def evaluate(y_true, y_score, y_pred):
    return {"pr_auc": float(average_precision_score(y_true, y_score)),
            "roc_auc": float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true))>1 else float("nan"),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "g_mean": g_mean(y_true, y_pred),
            "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
            "mcc": float(matthews_corrcoef(y_true, y_pred)) if len(np.unique(y_pred))>1 else 0.0,
            "total_cost": total_cost(y_true, y_pred)}


In [ ]:
# 5) Strategies over two AutoML engines (flaml / sklearn)
def build_preprocessor(X):
    num = X.select_dtypes(include=[np.number]).columns.tolist()
    cat = X.select_dtypes(exclude=[np.number]).columns.tolist()
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("ohe", OneHotEncoder(handle_unknown="ignore", max_categories=20, sparse_output=False))])
    return ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat)], remainder="drop")

def _fit_flaml(X_tr, y_tr, metric, time_budget, sample_weight, seed):
    try:
        from flaml import AutoML
    except ImportError:
        from flaml.automl.automl import AutoML
    m = AutoML()
    m.fit(X_train=X_tr, y_train=y_tr, task="classification", metric=metric, time_budget=time_budget,
          estimator_list=ESTIMATORS, sample_weight=sample_weight, seed=seed, verbose=0,
          early_stop=True, eval_method="cv", n_splits=3)
    return m

def _fit_sklearn(X_tr, y_tr, metric, sample_weight, seed, n_iter=SK_N_ITER):
    scoring = "average_precision" if metric == "ap" else "accuracy"
    base = HistGradientBoostingClassifier(random_state=seed, early_stopping=True)
    if sample_weight is not None:
        base.set_params(class_weight="balanced")
    dist = {"learning_rate":[0.01,0.03,0.05,0.1,0.2], "max_leaf_nodes":[15,31,63,127],
            "max_depth":[None,3,6,10], "l2_regularization":[0.0,0.1,1.0], "min_samples_leaf":[10,20,50]}
    rs = RandomizedSearchCV(base, dist, n_iter=n_iter, scoring=scoring, cv=3, error_score=0.0,
                            random_state=seed, n_jobs=-1, refit=True)
    rs.fit(X_tr, y_tr); return rs.best_estimator_

def _fit_xgb_gpu(X_tr, y_tr, metric, sample_weight, seed):
    from xgboost import XGBClassifier
    spw = 1.0
    if sample_weight is not None:                       # S3: balanced via scale_pos_weight
        npos = max(int(np.sum(y_tr == 1)), 1); nneg = int(np.sum(y_tr == 0)); spw = nneg / npos
    clf = XGBClassifier(tree_method="hist", device="cuda", n_estimators=600, learning_rate=0.05,
                        max_depth=6, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                        random_state=seed, n_jobs=-1, scale_pos_weight=spw,
                        eval_metric=("aucpr" if metric == "ap" else "logloss"),
                        early_stopping_rounds=30)
    # internal stratified split so the objective metric (aucpr for S2/S5) actually guides training
    Xa, Xb, ya, yb = train_test_split(X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=seed)
    clf.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return clf

def _fit(engine, X_tr, y_tr, metric, time_budget, sample_weight, seed, sk_n_iter=SK_N_ITER):
    if engine == "flaml":       return _fit_flaml(X_tr, y_tr, metric, time_budget, sample_weight, seed)
    if engine == "sklearn":     return _fit_sklearn(X_tr, y_tr, metric, sample_weight, seed, sk_n_iter)
    if engine == "xgboost_gpu": return _fit_xgb_gpu(X_tr, y_tr, metric, sample_weight, seed)
    raise ValueError(engine)

def _pos_proba(model, X):
    p = model.predict_proba(X); cl = list(model.classes_)
    return p[:, cl.index(1) if 1 in cl else p.shape[1]-1]

def tune_threshold(y_val, s_val, cost_fn=COST_FN, cost_fp=COST_FP):
    th = np.unique(np.concatenate([[0.0], np.sort(s_val), [1.0]])); best_t, best_c = 0.5, np.inf
    for t in th:
        pred = (s_val >= t).astype(int)
        c = int(np.sum((y_val==1)&(pred==0)))*cost_fn + int(np.sum((y_val==0)&(pred==1)))*cost_fp
        if c < best_c: best_c, best_t = c, t
    return float(best_t)

def run_strategy(strat, X_tr, y_tr, X_val, y_val, X_te, time_budget=30, seed=0, engine="flaml", sk_n_iter=SK_N_ITER):
    pre = build_preprocessor(X_tr)
    Xtr = pre.fit_transform(X_tr); Xval = pre.transform(X_val); Xte = pre.transform(X_te)
    metric, sw, use_smote, tune = "accuracy", None, False, False
    if strat == "S1_smote": use_smote = True
    elif strat == "S2_objective": metric = "ap"
    elif strat == "S3_classweight": sw = compute_sample_weight("balanced", y_tr)
    elif strat == "S4_threshold": tune = True
    elif strat == "S5_combined": metric, tune = "ap", True
    elif strat == "S6_full": use_smote, metric, tune = True, "ap", True
    ytr = y_tr
    if use_smote:
        from imblearn.over_sampling import SMOTE
        npos = int(np.sum(y_tr==1)); k = max(1, min(5, npos-1))
        if npos > 1:
            Xtr, ytr = SMOTE(random_state=seed, k_neighbors=k).fit_resample(Xtr, y_tr); sw = None
    model = _fit(engine, Xtr, ytr, metric, time_budget, sw, seed, sk_n_iter)
    s_val = _pos_proba(model, Xval); s_te = _pos_proba(model, Xte)
    thr = tune_threshold(y_val, s_val) if tune else 0.5
    return s_te, (s_te >= thr).astype(int), thr

In [ ]:
# 6) Resumable experiment runner
def split(X, y, seed):
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=seed)
    Xv, Xte, yv, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=seed)
    return Xtr, ytr, Xv, yv, Xte, yte

def run_experiment(datasets, strategies=STRATEGIES, engines=("flaml",), seeds=2,
                   time_budget=30, max_rows=120000, sk_iter=SK_N_ITER):
    rows, done = [], set()
    if os.path.exists(RESULTS_CSV):
        prev = pd.read_csv(RESULTS_CSV)
        if "engine" not in prev.columns: prev["engine"] = "flaml"
        rows = prev.to_dict("records")
        done = {(r["dataset"], r["strategy"], int(r["seed"]), r.get("engine","flaml")) for r in rows}
        print("[resume] %d combos already done" % len(done))
    for eng in engines:
        for ds in datasets:
            try:
                X, y, info = load_dataset(ds, max_rows=max_rows)
            except Exception as e:
                print("[skip]", ds, repr(e)[:120]); continue
            print("[data] %s (%s): %d rows, minority %.4f" % (ds, eng, info["n_rows"], info["minority_rate"]))
            for seed in range(seeds):
                Xtr, ytr, Xv, yv, Xte, yte = split(X, y, seed)
                for strat in strategies:
                    if (ds, strat, seed, eng) in done: continue
                    t0 = time.time()
                    try:
                        s_te, y_pred, thr = run_strategy(strat, Xtr, ytr, Xv, yv, Xte,
                                                         time_budget=time_budget, seed=seed, engine=eng, sk_n_iter=sk_iter)
                        m = evaluate(yte, s_te, y_pred)
                        m.update({"dataset":ds,"strategy":strat,"seed":seed,"engine":eng,
                                  "threshold":thr,"minority_rate":info["minority_rate"],"secs":round(time.time()-t0,1)})
                        rows.append(m)
                        print("  %-8s %-18s %-14s seed%d PRAUC=%.3f recall=%.3f cost=%.0f (%.1fs)" %
                              (eng, ds, strat, seed, m["pr_auc"], m["recall"], m["total_cost"], m["secs"]))
                        pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)   # checkpoint every fit
                    except Exception as e:
                        print("  [fail]", eng, ds, strat, "seed", seed, repr(e)[:120])
    print("DONE ->", RESULTS_CSV); return pd.DataFrame(rows)

In [ ]:
# 7) RUN — adjust these knobs
ALL = list(DATASETS)
df = run_experiment(
    datasets=ALL,             # or a subset e.g. ["credit_g","churn","adult_income"]
    strategies=STRATEGIES,
    engines=("xgboost_gpu", "flaml"),  # run BOTH engines (resumable) # GPU engine first, then FLAML. FLAML ignores GPU; lower time_budget if it is slow.
    seeds=10,                 # more seeds = tighter error bars (resumable: reuses earlier seeds)
    time_budget=60,           # seconds per AutoML fit (FLAML)
    max_rows=120000,          # fraud capped here; use 284807 for the full set
    sk_iter=25,               # sklearn-engine search size
)
print(df.shape)

In [ ]:
STRATEGIES=["S0_default","S1_smote","S2_objective","S3_classweight","S4_threshold","S5_combined","S6_full"]
HIGHER=["pr_auc","roc_auc","recall","precision","f1","g_mean","balanced_acc","mcc"]
LOWER=["total_cost"]; KEY=["pr_auc","f1","g_mean","total_cost"]
QA05={2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,8:3.031}

def _order_ds(d): return d.groupby("dataset")["minority_rate"].first().sort_values().index.tolist()
def _sv(fig_or_none, path): plt.tight_layout(); plt.savefig(path, dpi=130); plt.close()

def full_analysis(engine, ALL, FIG):
    d = ALL[ALL.engine==engine].copy() if "engine" in ALL.columns else ALL.copy()
    if len(d)==0: print("No rows for", engine); return None
    outdir=os.path.join(FIG,engine); os.makedirs(outdir,exist_ok=True)
    st=[s for s in STRATEGIES if s in set(d.strategy)]
    dss=_order_ds(d); mets=[m for m in HIGHER+LOWER if m in d.columns]
    mean_tbl=d.groupby("strategy")[mets].mean().reindex(st).round(4)
    mean_tbl.to_csv(os.path.join(outdir,"summary.csv"))
    print("=== [%s] mean metrics by strategy ==="%engine); print(mean_tbl.to_string())
    blocks=[]
    for _,g in d.groupby(["dataset","seed"]):
        blocks.append(pd.DataFrame({m:g.set_index("strategy")[m].rank(ascending=(m in LOWER)) for m in mets}))
    ranks=pd.concat(blocks).groupby(level=0).mean(); ranks["avg_rank_key"]=ranks[KEY].mean(axis=1)
    ranks=ranks.sort_values("avg_rank_key").round(3); ranks.to_csv(os.path.join(outdir,"ranks.csv"))
    print("\n[%s] avg rank (lower=better):"%engine); print(ranks["avg_rank_key"].to_string())
    piv=d.pivot_table(index=["dataset","seed"],columns="strategy",values="pr_auc").dropna()
    n_blocks=d.groupby(["dataset","seed"]).ngroups
    if piv.shape[0]>=2 and piv.shape[1]>=3:
        chi2,p=friedmanchisquare(*[piv[c].values for c in piv.columns])
        print("[%s] Friedman PR-AUC: chi2=%.3f p=%.4f (blocks=%d k=%d)"%(engine,chi2,p,piv.shape[0],piv.shape[1]))
    # CD diagram
    comp=[]
    for _,g in d.groupby(["dataset","seed"]):
        comp.append(pd.DataFrame({m:g.set_index("strategy")[m].rank(ascending=(m in LOWER)) for m in KEY}).mean(axis=1))
    avg=pd.concat(comp,axis=1).mean(axis=1).reindex(st).sort_values(); k=len(avg); q=QA05.get(k,3.0)
    CD=q*np.sqrt(k*(k+1)/(6.0*max(n_blocks,1)))
    fig,ax=plt.subplots(figsize=(9,3.2)); lo,hi=1,k
    ax.set_xlim(lo-0.5,hi+0.5); ax.set_ylim(0,1); ax.hlines(0.75,lo,hi,color="k"); ax.axis("off")
    for x in range(lo,hi+1): ax.vlines(x,0.73,0.77,color="k"); ax.text(x,0.80,str(x),ha="center",fontsize=9)
    for i,(nm,v) in enumerate(avg.items()):
        yy=0.55-(i%3)*0.13
        ax.plot([v,v],[0.73,yy],color="C0"); ax.plot([v,lo-0.4 if i<k/2 else hi+0.4],[yy,yy],color="C0")
        ax.text(lo-0.45 if i<k/2 else hi+0.45,yy,"%s (%.2f)"%(nm,v),ha="right" if i<k/2 else "left",va="center",fontsize=8)
    ax.hlines(0.93,lo,lo+CD,color="r"); ax.text(lo+CD/2,0.96,"CD=%.2f"%CD,ha="center",color="r",fontsize=8)
    ax.set_title("[%s] Critical-difference diagram (lower=better)"%engine); _sv(fig,os.path.join(outdir,"cd_diagram.png"))
    # bar plots
    for metric,lb in [("pr_auc",False),("total_cost",True),("f1",False),("g_mean",False)]:
        g=d.groupby("strategy")[metric]; mean,std=g.mean(),g.std(); order=mean.sort_values(ascending=lb).index
        plt.figure(figsize=(7,4)); plt.bar(range(len(order)),mean[order],yerr=std[order],capsize=4,color="#4C72B0")
        plt.xticks(range(len(order)),order,rotation=30,ha="right"); plt.ylabel(metric); plt.title("[%s] %s by strategy"%(engine,metric))
        _sv(None,os.path.join(outdir,"bar_%s.png"%metric))
    # grouped per dataset
    for metric,log in [("recall",False),("total_cost",True)]:
        pv=d.pivot_table(index="dataset",columns="strategy",values=metric,aggfunc="mean").reindex(dss)[st]
        x=np.arange(len(dss)); w=0.8/len(st); plt.figure(figsize=(12,5))
        for i,s in enumerate(st): plt.bar(x+i*w,pv[s].values,w,label=s)
        plt.xticks(x+0.4-w/2,dss,rotation=25,ha="right"); plt.ylabel(metric)
        if log: plt.yscale("log")
        plt.title("[%s] %s per dataset"%(engine,metric)); plt.legend(fontsize=8,ncol=len(st))
        _sv(None,os.path.join(outdir,"grouped_%s.png"%metric))
    # heatmaps
    for metric in ["pr_auc","recall","f1"]:
        pv=d.pivot_table(index="dataset",columns="strategy",values=metric,aggfunc="mean").reindex(dss)[st]
        plt.figure(figsize=(9,5)); im=plt.imshow(pv.values,cmap="RdYlGn",aspect="auto"); plt.colorbar(im,label=metric)
        plt.xticks(range(len(st)),st,rotation=25,ha="right"); plt.yticks(range(len(dss)),dss)
        for i in range(pv.shape[0]):
            for j in range(pv.shape[1]): plt.text(j,i,"%.2f"%pv.values[i,j],ha="center",va="center",fontsize=7)
        plt.title("[%s] %s heatmap"%(engine,metric)); _sv(None,os.path.join(outdir,"heatmap_%s.png"%metric))
    # recall vs cost scatter
    plt.figure(figsize=(7,5.5))
    for s in st:
        gg=d[d.strategy==s]; plt.scatter(gg["recall"].mean(),gg["total_cost"].mean(),s=90)
        plt.annotate(s,(gg["recall"].mean(),gg["total_cost"].mean()),fontsize=8,xytext=(4,4),textcoords="offset points")
    plt.xlabel("mean recall"); plt.ylabel("mean total cost"); plt.title("[%s] efficiency frontier"%engine); plt.grid(alpha=0.3)
    _sv(None,os.path.join(outdir,"recall_vs_cost.png"))
    # vs imbalance
    for metric,log in [("total_cost",True),("recall",False)]:
        plt.figure(figsize=(8,4.8))
        for s in st:
            gg=d[d.strategy==s].groupby("minority_rate")[metric].mean().sort_index()
            plt.plot(gg.index,gg.values,marker="o",label=s)
        plt.xscale("log")
        if log: plt.yscale("log")
        plt.xlabel("minority rate (log)"); plt.ylabel(metric); plt.title("[%s] %s vs imbalance"%(engine,metric)); plt.legend(fontsize=8)
        _sv(None,os.path.join(outdir,"%s_vs_imbalance.png"%metric))
    # boxplots
    for metric in ["pr_auc","f1","total_cost"]:
        data=[d[d.strategy==s][metric].values for s in st]
        plt.figure(figsize=(9,5)); bp=plt.boxplot(data,labels=st,patch_artist=True,showmeans=True)
        for pt in bp["boxes"]: pt.set_facecolor("#9ecae1")
        plt.xticks(rotation=25,ha="right"); plt.ylabel(metric); plt.title("[%s] %s distribution"%(engine,metric))
        _sv(None,os.path.join(outdir,"box_%s.png"%metric))
    # threshold
    m=d.groupby("strategy")["threshold"].mean().reindex(st)
    plt.figure(figsize=(8,4.5)); plt.bar(range(len(st)),m.values,color="#756bb1"); plt.axhline(0.5,color="k",ls="--",lw=0.8)
    plt.xticks(range(len(st)),st,rotation=25,ha="right"); plt.ylabel("mean tuned threshold"); plt.title("[%s] decision threshold"%engine)
    _sv(None,os.path.join(outdir,"threshold.png"))
    # latex table
    cols=[c for c in ["pr_auc","recall","f1","g_mean","mcc","total_cost"] if c in mean_tbl.columns]
    best={c:(mean_tbl[c].min() if c in LOWER else mean_tbl[c].max()) for c in cols}
    L=["\\begin{table}[t]\\centering","\\caption{Mean test metrics (%s engine).}"%engine,"\\begin{tabular}{l"+"r"*(len(cols)+1)+"}","\\toprule",
       "Strategy & "+" & ".join(c.replace('_',' ').upper() for c in cols)+" & Avg. rank \\\\","\\midrule"]
    for s in ranks.index:
        cells=["\\textbf{%.3f}"%mean_tbl.loc[s,c] if abs(mean_tbl.loc[s,c]-best[c])<1e-9 else "%.3f"%mean_tbl.loc[s,c] for c in cols]
        L.append(s.replace('_','\\_')+" & "+" & ".join(cells)+" & %.2f \\\\"%ranks.loc[s,"avg_rank_key"])
    L+=["\\bottomrule","\\end{tabular}","\\end{table}"]
    open(os.path.join(outdir,"results_table_%s.tex"%engine),"w").write("\n".join(L)+"\n")
    print("[%s] figures + tex saved to %s"%(engine,outdir))
    return mean_tbl, ranks

def compare_engines_full(ALL, FIG):
    if "engine" not in ALL.columns or ALL.engine.nunique()<2:
        print("compare: need >=2 engines"); return
    outdir=os.path.join(FIG,"comparison"); os.makedirs(outdir,exist_ok=True)
    engs=sorted(ALL.engine.unique())
    common=set.intersection(*[set(ALL[ALL.engine==e].dataset) for e in engs])
    d=ALL[ALL.dataset.isin(common)]; st=[s for s in STRATEGIES if s in set(d.strategy)]
    print("common datasets:",sorted(common))
    def avgrank(dd):
        bl=[]
        for _,g in dd.groupby(["dataset","seed"]):
            bl.append(pd.DataFrame({m:g.set_index("strategy")[m].rank(ascending=(m in LOWER)) for m in KEY}).mean(axis=1))
        return pd.concat(bl,axis=1).mean(axis=1)
    tab=pd.DataFrame({e:avgrank(d[d.engine==e]).reindex(st) for e in engs}); tab["mean_rank"]=tab.mean(axis=1)
    tab=tab.sort_values("mean_rank").round(3); tab.to_csv(os.path.join(outdir,"engine_comparison_ranks.csv"))
    print("=== cross-engine avg rank (lower=better) ==="); print(tab.to_string())
    print("best strategy per engine:",{e:tab[e].idxmin() for e in engs})
    for metric in ["pr_auc","recall","f1","total_cost"]:
        pv=d.pivot_table(index="strategy",columns="engine",values=metric,aggfunc="mean").reindex(st)
        x=np.arange(len(st)); w=0.8/len(engs); plt.figure(figsize=(11,5))
        for i,e in enumerate(engs): plt.bar(x+i*w,pv[e].values,w,label=e)
        plt.xticks(x+0.4-w/2,st,rotation=25,ha="right"); plt.ylabel(metric); plt.title("%s by strategy and engine"%metric); plt.legend()
        _sv(None,os.path.join(outdir,"cmp_%s.png"%metric))
    d.groupby(["engine","strategy"])[[m for m in HIGHER+LOWER if m in d.columns]].mean().round(4).to_csv(os.path.join(outdir,"per_engine_strategy_means.csv"))
    print("comparison saved to",outdir)

def curves_for_engine(engine):
    from sklearn.metrics import precision_recall_curve, roc_curve, auc, average_precision_score
    REP = ["credit_g", "churn", "adult_income"]
    outdir = os.path.join(FIG, engine); os.makedirs(outdir, exist_ok=True)
    cache = {}
    for ds in REP:
        X, y, _ = load_dataset(ds, max_rows=15000); Xtr,ytr,Xv,yv,Xte,yte = split(X,y,0)
        for s in STRATEGIES:
            s_te,_,_ = run_strategy(s, Xtr,ytr,Xv,yv,Xte, time_budget=20, seed=0, engine=engine)
            cache[(ds,s)] = (yte.copy(), s_te.copy())
        plt.figure(figsize=(7,5.5))
        for s in STRATEGIES:
            yt,sc = cache[(ds,s)]; p,r,_ = precision_recall_curve(yt,sc)
            plt.plot(r,p,label="%s (AP=%.2f)"%(s, average_precision_score(yt,sc)))
        plt.xlabel("recall"); plt.ylabel("precision"); plt.title("[%s] PR - %s"%(engine,ds)); plt.legend(fontsize=7)
        plt.tight_layout(); plt.savefig(os.path.join(outdir,"pr_curves_%s.png"%ds),dpi=130); plt.show()
        plt.figure(figsize=(7,5.5))
        for s in STRATEGIES:
            yt,sc = cache[(ds,s)]; fpr,tpr,_ = roc_curve(yt,sc)
            plt.plot(fpr,tpr,label="%s (AUC=%.2f)"%(s, auc(fpr,tpr)))
        plt.plot([0,1],[0,1],"k--",lw=0.8); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("[%s] ROC - %s"%(engine,ds)); plt.legend(fontsize=7)
        plt.tight_layout(); plt.savefig(os.path.join(outdir,"roc_curves_%s.png"%ds),dpi=130); plt.show()
    ratios=[1,2,3,5,10,20,50]
    fig,axes=plt.subplots(1,max(len(REP),1),figsize=(5*max(len(REP),1),4.5))
    if len(REP)<=1: axes=[axes]
    for ax,ds in zip(axes,REP):
        for s in STRATEGIES:
            yt,sc=cache[(ds,s)]; costs=[]
            for cfn in ratios:
                thr=tune_threshold(yt,sc,cost_fn=cfn,cost_fp=1.0); pred=(sc>=thr).astype(int)
                costs.append(int(np.sum((yt==1)&(pred==0)))*cfn+int(np.sum((yt==0)&(pred==1))))
            ax.plot(ratios,costs,marker="o",label=s)
        ax.set_xscale("log"); ax.set_xlabel("FN:FP cost ratio"); ax.set_ylabel("total cost"); ax.set_title(ds); ax.legend(fontsize=7)
    fig.suptitle("[%s] cost-ratio sensitivity"%engine); plt.tight_layout()
    plt.savefig(os.path.join(outdir,"cost_ratio_sensitivity.png"),dpi=130); plt.show()

In [ ]:
import pandas as pd
ALL = pd.read_csv(RESULTS_CSV)
print("engines in results:", sorted(ALL["engine"].unique()) if "engine" in ALL.columns else "none")
print(ALL.groupby(["engine","strategy"]).size() if "engine" in ALL.columns else len(ALL))

# Part A - xgboost_gpu: full analysis and all figures¶

In [ ]:
_ = full_analysis("xgboost_gpu", ALL, FIG)
curves_for_engine("xgboost_gpu")

# Part B - FLAML: full analysis and all figures

In [ ]:
_ = full_analysis("flaml", ALL, FIG)
curves_for_engine("flaml")

# Part C - Compare both engines (tables + figures)

In [ ]:
compare_engines_full(ALL, FIG)

In [ ]:
# Bundle every figure (png + pdf, all engines + comparison) into one downloadable zip
import shutil, os, glob
shutil.make_archive("/kaggle/working/figures", "zip", FIG)
print("PNG:", len(glob.glob(FIG+"/**/*.png", recursive=True)),
      "| PDF:", len(glob.glob(FIG+"/**/*.pdf", recursive=True)))
print("Folders:", [d for d in os.listdir(FIG) if os.path.isdir(os.path.join(FIG,d))])
print("Download: /kaggle/working/figures.zip")
